# Chapter 34: ARIMA and Exponential Smoothing

## Learning Objectives
- Apply differencing for stationarity
- Fit ARIMA(p,d,0) from scratch
- Build smoothing baseline
- Evaluate with horizon-aware metrics

## Prerequisites
- Strong understanding of chapters 0-29
- Python and basic linear algebra
- Numpy for probabilistic/deep chapters


# Chapter 34: ARIMA and Exponential Smoothing

## Why this chapter matters
Many enterprise forecasting tasks still rely on classical statistical models for interpretability, fast retraining, and limited-data robustness.

## Learning goals
1. Understand stationarity and differencing.
2. Implement ARIMA(p,d,0) baseline from scratch.
3. Implement simple exponential smoothing baseline.
4. Evaluate with walk-forward validation.

## ARIMA decomposition
ARIMA(p,d,q):
- `d`: differencing order to remove trend/non-stationarity
- `p`: autoregressive lag order
- `q`: moving average order

This chapter implements ARIMA(p,d,0) for clarity and extension.

## Workflow
1. Difference series `d` times.
2. Fit AR(p) coefficients with least squares.
3. Forecast future differenced values recursively.
4. Invert differencing to return original scale.

## Exponential smoothing
Simple exponential smoothing update:
\[
\hat{y}_{t+1} = \alpha y_t + (1-\alpha)\hat{y}_t
\]

## Industry notes
- Always compare against naive and seasonal-naive baseline.
- Use rolling backtests, never random split.
- Track forecast bias and horizon-wise error.

## Assignment
1. Tune AR order `p` and differencing `d`.
2. Compare ARIMA-like model vs smoothing baseline.
3. Report MAE and sMAPE by horizon buckets.


In [ ]:
from __future__ import annotations

from typing import List

import numpy as np


def difference(series: List[float], d: int) -> List[float]:
    out = series[:]
    for _ in range(d):
        out = [out[i] - out[i - 1] for i in range(1, len(out))]
    return out


def invert_difference(history: List[float], diff_forecasts: List[float], d: int) -> List[float]:
    if d == 0:
        return diff_forecasts[:]
    preds = []
    base = history[-1]
    for val in diff_forecasts:
        base = base + val
        preds.append(base)
    return preds


class ARIMAZeroMA:
    """ARIMA(p,d,0) educational implementation."""

    def __init__(self, p: int = 3, d: int = 1):
        self.p = p
        self.d = d
        self.coef = None

    def fit(self, series: List[float]) -> None:
        ds = difference(series, self.d)
        if len(ds) <= self.p:
            raise ValueError("Series too short for given p and d.")

        X, y = [], []
        for t in range(self.p, len(ds)):
            X.append(ds[t - self.p : t])
            y.append(ds[t])
        X = np.asarray(X)
        y = np.asarray(y)

        self.coef, *_ = np.linalg.lstsq(X, y, rcond=None)

    def forecast(self, series: List[float], horizon: int) -> List[float]:
        ds = difference(series, self.d)
        hist = ds[:]
        diff_preds = []

        for _ in range(horizon):
            x = np.asarray(hist[-self.p :])
            pred = float(x @ self.coef)
            hist.append(pred)
            diff_preds.append(pred)

        return invert_difference(series, diff_preds, self.d)


class SimpleExponentialSmoothing:
    def __init__(self, alpha: float = 0.3):
        self.alpha = alpha
        self.level = None

    def fit(self, series: List[float]) -> None:
        level = series[0]
        for y in series[1:]:
            level = self.alpha * y + (1 - self.alpha) * level
        self.level = level

    def forecast(self, horizon: int) -> List[float]:
        return [self.level] * horizon


def mae(y_true: List[float], y_pred: List[float]) -> float:
    return sum(abs(a - b) for a, b in zip(y_true, y_pred)) / len(y_true)


if __name__ == "__main__":
    series = [10, 12, 13, 16, 18, 20, 23, 25, 27, 29, 31, 33]
    train, test = series[:-3], series[-3:]

    ar = ARIMAZeroMA(p=2, d=1)
    ar.fit(train)
    ar_pred = ar.forecast(train, horizon=3)

    ses = SimpleExponentialSmoothing(alpha=0.4)
    ses.fit(train)
    ses_pred = ses.forecast(3)

    print("ARIMA-like pred:", [round(v, 3) for v in ar_pred], "MAE:", round(mae(test, ar_pred), 3))
    print("SES pred      :", [round(v, 3) for v in ses_pred], "MAE:", round(mae(test, ses_pred), 3))


## Checkpoint

1. You can explain purpose of differencing.
2. You can train AR coefficients with least squares.
3. You can compare ARIMA-like and SES baselines.

## Guided Exercise
Backtest multiple `(p,d)` combinations and compare MAE over rolling splits.

In [ ]:
# TODO (Student Exercise)
series = [10, 12, 13, 16, 18, 20, 23, 25, 27, 29, 31, 33]
for p, d in [(1,1), (2,1), (3,1)]:
    m = ARIMAZeroMA(p=p, d=d)
    m.fit(series[:-3])
    pred = m.forecast(series[:-3], 3)
    print((p,d), round(mae(series[-3:], pred), 3), pred)

## Quick Quiz

**Q1.** Why avoid random train/test split in time series?

**Answer:** It leaks future information and breaks temporal causality.

**Q2.** What does `d` control in ARIMA?

**Answer:** Differencing order to help stationarity.

**Q3.** Why keep SES as baseline?

**Answer:** Simple methods are hard to beat and expose overfitting quickly.


## Homework
Add walk-forward evaluation and horizon-bucket errors (`t+1`, `t+3`, `t+7`).